# metabeta — introductory demo

This notebook demonstrates the end-to-end inference workflow:
1. Load a real hierarchical dataset
2. Specify a mixed-effects formula
3. Package model checkpoints into a joint checkpoint
4. Run amortized posterior inference with `Router`

In [1]:
from pathlib import Path
import pandas as pd
import torch

from metabeta.models.router import Router
from metabeta.utils.router import joinCheckpoints

## 1. Load the dataset

The `sleepstudy` dataset (Belenky et al. 2003) measures reaction times (ms) for 18 subjects over 10 days of sleep restriction. It is a classic two-level dataset: observations nested within subjects.

In [2]:
DATASET_PATH = Path('../metabeta/datasets/from-r/parquet/sleep.parquet')

df = pd.read_parquet(DATASET_PATH)
print(f'{len(df)} observations, {df["group"].nunique()} subjects')
df.head()

180 observations, 18 subjects


,y,Days,group
0,249.5600,0.0,308
1,258.7047,1.0,308
2,250.8006,2.0,308
3,321.4398,3.0,308
4,356.8519,4.0,308


## 2. Specify a formula

We use a standard lme4-style formula. `Days` enters as a fixed slope; a random intercept is estimated per subject.

In [3]:
FORMULA = 'y ~ Days + (1 | group)'

## 3. Build a joint checkpoint

`Router` expects a *joint checkpoint* — a single `.pt` file that bundles one or more trained submodels with their routing metadata. `joinCheckpoints` assembles it from individual training checkpoints.

Here we use a single `small-n-mixed` submodel, trained on continuous (normally-distributed) outcomes.

In [4]:
CHECKPOINT_DIR = Path('../metabeta/outputs/checkpoints')
JOINT_PATH = Path('/tmp/joint_normal_v1.pt')

joinCheckpoints(
    {'small-n-s1': CHECKPOINT_DIR / 'data=small-n-mixed_model=large_seed=1'},
    output_path=JOINT_PATH,
)
print('Joint checkpoint written to', JOINT_PATH)

Joint checkpoint written to /tmp/joint_normal_v1.pt


## 4. Load the Router

`Router` loads the joint checkpoint and lazily instantiates submodels on first use.

In [5]:
router = Router(JOINT_PATH, device='cpu')
print('Submodels:', [s['id'] for s in router.submodels])

Submodels: ['small-n-s1']


## 5. Run inference

`router.sample()` accepts a raw DataFrame. It runs the `DataPreprocessor` internally (`fit_preprocessor=True`), builds design matrices from the formula, routes the dataset to the compatible submodel, and returns posterior samples.

Fixed-effect samples (`ffx`) are in standardized units (the preprocessor z-scores `y`). Random-effect standard deviations (`sigma_rfx`) and residual SD (`sigma_eps`) are rescaled back to the original scale of `y`.

In [6]:
result = router.sample(
    df,
    formula=FORMULA,
    fit_preprocessor=True,
    n_samples=1000,
)

print('Routed to submodel:', result.routes)
print('ffx samples shape  (B, S, d):', result.proposal.ffx.shape)
print('rfx samples shape  (B, m, S, q):', result.proposal.rfx.shape)
print('sigma_rfx shape    (B, S, q):', result.proposal.sigma_rfx.shape)
print('sigma_eps shape    (B, S):', result.proposal.sigma_eps.shape)

Routed to submodel: ['small-n-s1']
ffx samples shape  (B, S, d): torch.Size([1, 1000, 4])
rfx samples shape  (B, m, S, q): torch.Size([1, 18, 1000, 2])
sigma_rfx shape    (B, S, q): torch.Size([1, 1000, 2])
sigma_eps shape    (B, S): torch.Size([1, 1000])


## 6. Inspect posterior summaries

In [7]:
print(router.posteriorSummary(result))

Formula:  y ~ Days + (1 | group)
Priors:   default

Fixed Effects:
|           |   Mean |     SD |    2.5% |   97.5% |   P(>0) |
|:----------|-------:|-------:|--------:|--------:|--------:|
| Intercept | -0.402 | 10.740 | -20.516 |  21.042 |   0.482 |
| days      | 30.348 |  4.667 |  21.586 |  40.122 |   1.000 |

Standard Deviations:
|           |   Mean |    SD |   2.5% |   97.5% |
|:----------|-------:|------:|-------:|--------:|
| Intercept | 39.707 | 7.959 | 27.643 |  57.161 |
| Residual  | 31.125 | 1.750 | 27.782 |  34.715 |

n_samples = 1000
